Task #4: Apply crosswalk to AIOE & Frey-Osborne
Staneisha Chambers 9/15/26

AIOE and Frey-Osborne both use six-digit 2010 SOC occupation codes.
Because the rest of the project is being aligned to 2018 SOC codes,
each dataset must be joined separately to the cleaned BLS 2010-to-2018 crosswalk.

AIOE source rows: 774
Frey-Osborne machine-readable rows: 653

## Task 4 Summary

### AIOE
- Source rows: 774
- Matched source rows: 773
- Unmatched source rows: 1
- Match rate: 99.87%
- Rows after crosswalk expansion: 827
- Unmatched occupation: 19-1020, Biologists
- Likely cause: SOC granularity mismatch. 19-1020 is a broad 2010 SOC occupation, while the crosswalk contains its detailed child occupations.

### Frey-Osborne
- Source rows: 653
- Matched source rows: 653
- Unmatched source rows: 0
- Match rate: 100%
- Rows after crosswalk expansion: 681

### Split handling
For 1-to-many SOC mappings, the original AIOE or Frey-Osborne score is copied to each mapped 2018 occupation rather than divided. The scores are exposure/probability measures rather than additive quantities.

### Deliverables
- `data/datasets/processed/aioe_2018soc.csv`
- `data/datasets/processed/frey_osborne_2018soc.csv`

In [1]:
import pandas as pd
import numpy as np

In [2]:
AIOE = pd.read_excel("../data/datasets/AIOE_appendixA.xlsx")
FREY =pd.read_csv("../data/datasets/frey_osborne_probabilities.csv")
CROSSWALK = pd.read_csv("../data/datasets/processed/crosswalk_clean.csv")

In [5]:
AIOE[["SOC Code", "Occupation Title"]].head()


,SOC Code,Occupation Title
0,11-1011,Chief Executives
1,11-1021,General and Operations Managers
2,11-2011,Advertising and Promotions Managers
3,11-2021,Marketing Managers
4,11-2022,Sales Managers


In [6]:
CROSSWALK[["2010 SOC Code", "2010 SOC Title"]].head()


,2010 SOC Code,2010 SOC Title
0,11-1011,Chief Executives
1,11-1021,General and Operations Managers
2,11-1031,Legislators
3,11-2011,Advertising and Promotions Managers
4,11-2021,Marketing Managers


In [7]:
AIOE["SOC Code"].dtype

<StringDtype(storage='python', na_value=nan)>

In [20]:
FREY['soc_code_2010'].dtype

<StringDtype(storage='python', na_value=nan)>

In [8]:
CROSSWALK["2010 SOC Code"].dtype

<StringDtype(storage='python', na_value=nan)>

In [10]:
aioe_merged = AIOE.merge(
    CROSSWALK,
    how="left",
    left_on="SOC Code",
    right_on="2010 SOC Code"
)

In [11]:
aioe_merged.shape

(827, 10)

In [12]:
aioe_merged.head()

,SOC Code,Occupation Title,AIOE,2010 SOC Code,2010 SOC Title,2018 SOC Code,2018 SOC Title,has_footnote_marker,match_type,handling
0,11-1011,Chief Executives,1.334246,11-1011,Chief Executives,11-1011,Chief Executives,False,1:1,1:1
1,11-1021,General and Operations Managers,0.574877,11-1021,General and Operations Managers,11-1021,General and Operations Managers,False,1:1,1:1
2,11-2011,Advertising and Promotions Managers,1.294387,11-2011,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers,False,1:1,1:1
3,11-2021,Marketing Managers,1.315032,11-2021,Marketing Managers,11-2021,Marketing Managers,False,1:1,1:1
4,11-2022,Sales Managers,1.266280,11-2022,Sales Managers,11-2022,Sales Managers,False,1:1,1:1


In [17]:
aioe_match = AIOE["SOC Code"].isin(CROSSWALK["2010 SOC Code"])
aioe_match.value_counts()


SOC Code
True     773
False      1
Name: count, dtype: int64

In [16]:
aioe_match.mean()

np.float64(0.9987080103359173)

In [18]:
AIOE[aioe_match == False]

,SOC Code,Occupation Title,AIOE
120,19-1020,Biologists,0.752081


In [19]:
CROSSWALK[
    CROSSWALK["2010 SOC Code"].str.startswith("19-102")
]

,2010 SOC Code,2010 SOC Title,2018 SOC Code,2018 SOC Title,has_footnote_marker,match_type,handling
142,19-1021,Biochemists and Biophysicists,19-1021,Biochemists and Biophysicists,False,1:1,1:1
143,19-1022,Microbiologists,19-1022,Microbiologists,False,1:1,1:1
144,19-1023,Zoologists and Wildlife Biologists,19-1023,Zoologists and Wildlife Biologists,False,1:1,1:1
145,19-1029,"Biological Scientists, All Other",19-1029,"Biological Scientists, All Other",False,1:1,1:1


In [21]:
frey_merged = FREY.merge(
    CROSSWALK,
    how="left",
    left_on="soc_code_2010",
    right_on="2010 SOC Code"
)

In [22]:
frey_merged.shape
frey_merged.head()

,soc_code_2010,fo_computerization_probability,2010 SOC Code,2010 SOC Title,2018 SOC Code,2018 SOC Title,has_footnote_marker,match_type,handling
0,11-1011,0.015,11-1011,Chief Executives,11-1011,Chief Executives,False,1:1,1:1
1,11-1021,0.160,11-1021,General and Operations Managers,11-1021,General and Operations Managers,False,1:1,1:1
2,11-2011,0.039,11-2011,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers,False,1:1,1:1
3,11-2021,0.014,11-2021,Marketing Managers,11-2021,Marketing Managers,False,1:1,1:1
4,11-2022,0.013,11-2022,Sales Managers,11-2022,Sales Managers,False,1:1,1:1


In [ ]:
frey_match = FREY["soc_code_2010"].isin(CROSSWALK["2010 SOC Code"])
frey_match.value_counts()

soc_code_2010
True    653
Name: count, dtype: int64

In [25]:
frey_match.mean()

np.float64(1.0)

In [27]:
FREY[frey_match == False]

,soc_code_2010,fo_computerization_probability


In [28]:
frey_merged.shape

(681, 9)

In [29]:
aioe_merged.columns

Index(['SOC Code', 'Occupation Title', 'AIOE', '2010 SOC Code',
       '2010 SOC Title', '2018 SOC Code', '2018 SOC Title',
       'has_footnote_marker', 'match_type', 'handling'],
      dtype='str')

In [31]:
aioe_final = aioe_merged[
    [
        "SOC Code",
        "Occupation Title",
        "AIOE",
        "2018 SOC Code",
        "2018 SOC Title",
        "has_footnote_marker",
        "match_type",
        "handling"
    ]
].rename(columns={
    "SOC Code": "soc2010",
    "Occupation Title": "title_2010",
    "AIOE": "aioe",
    "2018 SOC Code": "soc2018",
    "2018 SOC Title": "title_2018"
})

In [30]:
frey_merged.columns

Index(['soc_code_2010', 'fo_computerization_probability', '2010 SOC Code',
       '2010 SOC Title', '2018 SOC Code', '2018 SOC Title',
       'has_footnote_marker', 'match_type', 'handling'],
      dtype='str')

In [32]:
frey_final = frey_merged[
    [
        "soc_code_2010",
        "2010 SOC Title",
        "fo_computerization_probability",
        "2018 SOC Code",
        "2018 SOC Title",
        "has_footnote_marker",
        "match_type",
        "handling"
    ]
].rename(columns={
    "soc_code_2010": "soc2010",
    "2010 SOC Title": "title_2010",
    "fo_computerization_probability": "fo_prob",
    "2018 SOC Code": "soc2018",
    "2018 SOC Title": "title_2018"
})

In [33]:
aioe_final.head()

,soc2010,title_2010,aioe,soc2018,title_2018,has_footnote_marker,match_type,handling
0,11-1011,Chief Executives,1.334246,11-1011,Chief Executives,False,1:1,1:1
1,11-1021,General and Operations Managers,0.574877,11-1021,General and Operations Managers,False,1:1,1:1
2,11-2011,Advertising and Promotions Managers,1.294387,11-2011,Advertising and Promotions Managers,False,1:1,1:1
3,11-2021,Marketing Managers,1.315032,11-2021,Marketing Managers,False,1:1,1:1
4,11-2022,Sales Managers,1.266280,11-2022,Sales Managers,False,1:1,1:1


In [34]:
frey_final.head()

,soc2010,title_2010,fo_prob,soc2018,title_2018,has_footnote_marker,match_type,handling
0,11-1011,Chief Executives,0.015,11-1011,Chief Executives,False,1:1,1:1
1,11-1021,General and Operations Managers,0.160,11-1021,General and Operations Managers,False,1:1,1:1
2,11-2011,Advertising and Promotions Managers,0.039,11-2011,Advertising and Promotions Managers,False,1:1,1:1
3,11-2021,Marketing Managers,0.014,11-2021,Marketing Managers,False,1:1,1:1
4,11-2022,Sales Managers,0.013,11-2022,Sales Managers,False,1:1,1:1


In [35]:
aioe_final.shape, frey_final.shape

((827, 8), (681, 8))

In [36]:
aioe_final.to_csv(
    "../data/datasets/processed/aioe_2018soc.csv",
    index=False
)

frey_final.to_csv(
    "../data/datasets/processed/frey_osborne_2018soc.csv",
    index=False
)

In [37]:
aioe_check = pd.read_csv("../data/datasets/processed/aioe_2018soc.csv")
frey_check = pd.read_csv("../data/datasets/processed/frey_osborne_2018soc.csv")

aioe_check.shape, frey_check.shape

((827, 8), (681, 8))

In [38]:
aioe_check["soc2018"].isna().sum()

np.int64(1)

In [39]:
frey_check["soc2018"].isna().sum()

np.int64(0)